**Starting With Combined Media (Twitter & Article Signals)**

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_absolute_error
from statsmodels.tsa.stattools import adfuller
import re
import ast
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch.utils.data as data
import random
import copy
import torch
from sklearn.model_selection import TimeSeriesSplit
import torch.nn as nn

In [48]:
def adf_test(series, name):

    result = adfuller(series)

    print("="*60)
    print(name)
    print(f"ADF Statistic : {result[0]:.4f}")
    print(f"P-value       : {result[1]:.4f}")

    if result[1] < 0.05:    
        print("Stationary")
    else:
        print("Non-stationary")

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def Evaluator(actual, prediction):
    # Convert to Series so we can align/drop NaNs
    actual = pd.Series(actual).reset_index(drop=True)
    prediction = pd.Series(prediction).reset_index(drop=True)

    # Combine into one dataframe
    results = pd.DataFrame({
        "actual": actual,
        "prediction": prediction
    })

    # Remove NaN rows
    results = results.dropna()

    # Calculate metrics
    mae = mean_absolute_error(
        results["actual"],
        results["prediction"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            results["actual"],
            results["prediction"]
        )
    )

    r2 = r2_score(
        results["actual"],
        results["prediction"]
    )

    print(f"Observations used: {len(results)}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}")

In [49]:
California= pd.read_csv('../data/interim/Media_with_features.csv')

In [50]:
California.head()

,year_month,article_count,rate_per_100k,mean_title_sentiment,mean_title_risk,mean_body_risk,avg_sentiment,avg_risk,mean_body_sentiment,tweet_count,avg_likes
0,2011-02,1.0,0.715017,0.028871,0.000000,0.015097,-0.465573,0.047619,-0.273456,6,0.000000
1,2011-03,1.0,0.821057,0.021148,0.000000,0.005263,-0.012873,0.000000,0.198157,5,1.000000
2,2011-04,0.0,0.833176,-0.125413,0.036783,0.020489,-0.185547,0.021802,-0.167308,0,7.516662
3,2011-05,0.0,0.875592,-0.125413,0.036783,0.020489,-0.185547,0.021802,-0.167308,0,7.516662
4,2011-06,0.0,0.969514,-0.125413,0.036783,0.020489,-0.185547,0.021802,-0.167308,0,7.516662


In [51]:
California.columns

Index(['year_month', 'article_count', 'rate_per_100k', 'mean_title_sentiment',
       'mean_title_risk', 'mean_body_risk', 'avg_sentiment', 'avg_risk',
       'mean_body_sentiment', 'tweet_count', 'avg_likes'],
      dtype='str')

In [52]:
cols =['article_count', 'rate_per_100k', 'mean_title_sentiment',
       'mean_title_risk', 'mean_body_risk', 'avg_sentiment', 'avg_risk',
       'mean_body_sentiment', 'tweet_count', 'avg_likes']

for col in cols:
    adf_test(California[col], col)

adf_test(California['rate_per_100k'], "VF Rate per 100k")

article_count
ADF Statistic : -7.0902
P-value       : 0.0000
Stationary
rate_per_100k
ADF Statistic : -0.2003
P-value       : 0.9385
Non-stationary
mean_title_sentiment
ADF Statistic : -11.0878
P-value       : 0.0000
Stationary
mean_title_risk
ADF Statistic : -12.3249
P-value       : 0.0000
Stationary
mean_body_risk
ADF Statistic : -10.8205
P-value       : 0.0000
Stationary
avg_sentiment
ADF Statistic : -4.3870
P-value       : 0.0003
Stationary
avg_risk
ADF Statistic : -3.4830
P-value       : 0.0084
Stationary
mean_body_sentiment
ADF Statistic : -13.7563
P-value       : 0.0000
Stationary
tweet_count
ADF Statistic : -2.3188
P-value       : 0.1660
Non-stationary
avg_likes
ADF Statistic : -2.4387
P-value       : 0.1311
Non-stationary
VF Rate per 100k
ADF Statistic : -0.2003
P-value       : 0.9385
Non-stationary


In [53]:
California['log_rate'] = np.log1p(
    California['rate_per_100k']
)
adf_test(California['log_rate'], "VF Rate per 100k")

California['logdiff_rate']= California['log_rate'].diff()
California= California.dropna(subset= 'logdiff_rate')
adf_test(California['logdiff_rate'], 'VF logdiffed rates')

VF Rate per 100k
ADF Statistic : -0.3003
P-value       : 0.9255
Non-stationary
VF logdiffed rates
ADF Statistic : -8.1790
P-value       : 0.0000
Stationary


In [54]:
threem= 163
onem = 165
sixm=160

In [55]:
train1 = California.iloc[:onem]
test1 = California.iloc[onem:]

train3 = California.iloc[:threem]
test3 = California.iloc[threem:]

train6 = California.iloc[:sixm]
test6 = California.iloc[sixm:]

In [56]:
testcols= ['article_count','mean_title_sentiment','mean_title_risk','mean_body_risk','avg_sentiment','avg_risk','mean_body_sentiment', 'tweet_count','avg_likes']
artcols= ['article_count','mean_title_sentiment','mean_title_risk','mean_body_risk']
twcols= ['avg_sentiment','avg_risk','mean_body_sentiment','tweet_count','avg_likes']

In [57]:
def create_dataset(dataset, lookback):
    X, y = [], []

    for i in range(len(dataset) - lookback):
        
        # Past log_rate + exogenous variables
        X.append(
            dataset[i:i+lookback, :]
        )
        
        # Next log_rate value (column 0)
        y.append(
            dataset[i+lookback, 0]
        )

    return (
        torch.tensor(
            np.array(X),
            dtype=torch.float32
        ),
        torch.tensor(
            np.array(y),
            dtype=torch.float32
        ).reshape(-1, 1)
    )

In [58]:
class LSTMModel(nn.Module):
    def __init__(
        self,
        n_features,
        hidden_size=50,
        num_layers=1,
        dropout=0
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.linear = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.linear(x)

        return x
loss_fn = nn.MSELoss()

In [59]:
pairs = [
    ("1_month", train1, test1),
    ("3_month", train3, test3),
    ("6_month", train6, test6)
]

for horizon, train, test in pairs:

    # -----------------------------
    # 1. Split TRAIN into
    #    actual training + validation
    # -----------------------------

    split_idx = int(len(train) * 0.80)

    train_part = train.iloc[:split_idx]
    val_part = train.iloc[split_idx:]

    # -----------------------------
    # 2. Numeric columns only
    # -----------------------------

    feature_cols = train.select_dtypes(
        include=np.number
    ).columns

    # -----------------------------
    # 3. Fit scaler ONLY on train
    # -----------------------------

    scaler = MinMaxScaler()

    train_scaled = scaler.fit_transform(
        train_part[feature_cols]
    )

    val_scaled = scaler.transform(
        val_part[feature_cols]
    )

    test_scaled = scaler.transform(
        test[feature_cols]
    )

In [60]:
results = []
for lookback in [3, 6, 12]:
    for hidden_size in [16, 32, 64]:
        for num_layers in [1, 2]:
            for lr in [0.001, 0.0001]:
                for batch_size in [8, 16]:

                    # 1. BUILD TRAIN SEQUENCES
                   
                    X_train, y_train = create_dataset(
                        train_scaled, lookback
                    )

                    # 2. BUILD VALIDATION SEQUENCES
                    val_input = np.concatenate(
                        [train_scaled[-lookback:], val_scaled],
                        axis=0
                    )

                    X_val, y_val = create_dataset(
                        val_input, lookback
                    )

                    # -----------------------------
                    # 3. CONVERT TO TENSORS
                    # -----------------------------
                    X_train_t = torch.tensor(
                        X_train, dtype=torch.float32
                    )
                    y_train_t = torch.tensor(
                        y_train, dtype=torch.float32
                    ).reshape(-1, 1)

                    X_val_t = torch.tensor(
                        X_val, dtype=torch.float32
                    )
                    y_val_t = torch.tensor(
                        y_val, dtype=torch.float32
                    ).reshape(-1, 1)

                    # -----------------------------
                    # 4. BUILD MODEL
                    # -----------------------------
                    model = LSTMModel(
                        n_features=X_train.shape[2],
                        hidden_size=hidden_size,
                        num_layers=num_layers
                    )

                    # IMPORTANT:
                    # recreate optimizer for every
                    # hyperparameter combination
                    optimizer = torch.optim.Adam(
                        model.parameters(),
                        lr=lr
                    )

                    # -----------------------------
                    # 5. DATA LOADER
                    # -----------------------------
                    train_loader = data.DataLoader(
                        data.TensorDataset(
                            X_train_t,
                            y_train_t
                        ),
                        batch_size=batch_size,
                        shuffle=True
                    )

                    # -----------------------------
                    # 6. TRAIN
                    # -----------------------------
                    best_val_rmse = np.inf
                    patience = 20
                    patience_counter = 0
                    best_epoch = 0
                    best_weights = None
                    max_epochs = 300

                    for epoch in range(max_epochs):

                        model.train()

                        for X_batch, y_batch in train_loader:

                            optimizer.zero_grad()

                            y_pred = model(X_batch)

                            loss = loss_fn(
                                y_pred,
                                y_batch
                            )

                            loss.backward()
                            optimizer.step()

                        # -------------------------
                        # VALIDATION
                        # -------------------------
                        model.eval()

                        with torch.no_grad():

                            val_pred = model(X_val_t)

                            val_rmse = torch.sqrt(
                                torch.mean(
                                    (val_pred - y_val_t) ** 2
                                )
                            ).item()

                        # -------------------------
                        # EARLY STOPPING
                        # -------------------------
                        if val_rmse < best_val_rmse:

                            best_val_rmse = val_rmse
                            patience_counter = 0
                            best_epoch = epoch + 1

                            best_weights = copy.deepcopy(
                                model.state_dict()
                            )

                        else:

                            patience_counter += 1

                        if patience_counter >= patience:
                            break

                    # Restore best validation model
                    model.load_state_dict(best_weights)

                    # -----------------------------
                    # 7. SAVE VALIDATION SCORE
                    # -----------------------------
                    results.append({
                        "lookback": lookback,
                        "hidden_size": hidden_size,
                        "num_layers": num_layers,
                        "lr": lr,
                        "batch_size": batch_size,
                        "best_epoch": best_epoch,
                        "val_rmse": best_val_rmse
                    })

                    print(
                        f"Lookback={lookback}, "
                        f"Hidden={hidden_size}, "
                        f"Layers={num_layers}, "
                        f"LR={lr}, "
                        f"Batch={batch_size}, "
                        f"Best Epoch={best_epoch} | "
                        f"Val RMSE={best_val_rmse:.4f}"
                    )


C:\Users\zions\AppData\Local\Temp\ipykernel_6340\3005663009.py:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_t = torch.tensor(
C:\Users\zions\AppData\Local\Temp\ipykernel_6340\3005663009.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_t = torch.tensor(
C:\Users\zions\AppData\Local\Temp\ipykernel_6340\3005663009.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val_t = torch.tensor(
C:\Users\zions\AppData\Local\Temp\ipykernel_6340\3005663009.py:37: UserWarning: To copy construct from a tensor, it is recommende

Lookback=3, Hidden=16, Layers=1, LR=0.001, Batch=8, Best Epoch=94 | Val RMSE=0.6132
Lookback=3, Hidden=16, Layers=1, LR=0.001, Batch=16, Best Epoch=79 | Val RMSE=0.6320
Lookback=3, Hidden=16, Layers=1, LR=0.0001, Batch=8, Best Epoch=247 | Val RMSE=0.6453
Lookback=3, Hidden=16, Layers=1, LR=0.0001, Batch=16, Best Epoch=297 | Val RMSE=0.6513


KeyboardInterrupt: 

In [61]:
results_df = pd.DataFrame(results)

summary = (
    results_df
    .groupby(
        [
            "lookback",
            "hidden_size",
            "num_layers",
            "lr",
            "batch_size"
        ]
    )
    .agg(
        mean_rmse=("val_rmse", "mean"),
        std_rmse=("val_rmse", "std"),
        mean_best_epoch=("best_epoch", "mean")
    )
    .reset_index()
    .sort_values("mean_rmse")
)

summary.head(10)

,lookback,hidden_size,num_layers,lr,batch_size,mean_rmse,std_rmse,mean_best_epoch
2,3,16,1,0.0010,8,0.613230,NaN,94.0
3,3,16,1,0.0010,16,0.632047,NaN,79.0
0,3,16,1,0.0001,8,0.645265,NaN,247.0
1,3,16,1,0.0001,16,0.651332,NaN,297.0


In [62]:
lookback=3
hidden_size=64
num_layers=2
lr=.001
batch_size=8
epoch=28

In [63]:
model = LSTMModel(
    n_features=train_scaled.shape[1],
    hidden_size=64,
    num_layers=2
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [64]:
X_train, y_train = create_dataset(
    train_scaled,
    lookback=3
)

X_train_t = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_t = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1, 1)

train_loader = data.DataLoader(
    data.TensorDataset(X_train_t, y_train_t),
    batch_size=8,
    shuffle=True
)

C:\Users\zions\AppData\Local\Temp\ipykernel_6340\1463001979.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_t = torch.tensor(
C:\Users\zions\AppData\Local\Temp\ipykernel_6340\1463001979.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_t = torch.tensor(


In [65]:
for epoch in range(28):

    model.train()

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        y_pred = model(X_batch)

        loss = loss_fn(y_pred, y_batch)

        loss.backward()

        optimizer.step()

In [66]:
test_input = np.concatenate(
    [train_scaled[-3:], test_scaled],
    axis=0
)

X_test, y_test = create_dataset(
    test_input,
    lookback=3
)

X_test_t = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_t = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

model.eval()

with torch.no_grad():
    predictions = model(X_test_t)

C:\Users\zions\AppData\Local\Temp\ipykernel_6340\2568384658.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_t = torch.tensor(
C:\Users\zions\AppData\Local\Temp\ipykernel_6340\2568384658.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test_t = torch.tensor(


In [67]:


pred = predictions.numpy().flatten()
actual = y_test_t.numpy().flatten()

rmse = rmse(actual, pred)
mae = mean_absolute_error(actual, pred)
r2 = r2_score(actual, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

RMSE: 1.2258741549020933
MAE: 0.9341244101524353
R²: -0.6718055009841919
